<a href="https://colab.research.google.com/github/EmanHrustemovic/FlyRank-AI-Intership-ML-Track-/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

My rule (revised): flag pages where observed CTR underperforms the expected
CTR for their position bucket (derived from Signal 2). This is the CTR-fix
logic from the session, applied to my lane.

Reason codes:
- CTR_UNDERPERFORM: CTR significantly below the expected value for its
  position bucket
- HEALTHY: CTR at or above expected

Action labels:
- "fix-ctr": CTR_UNDERPERFORM
- "monitor": HEALTHY

In [17]:
import duckdb
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get('eman')
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

**Signal 1 —** **Staleness vs decline (bucket table)**:


In [18]:
signal1 = con.sql(f"""
    WITH daily AS (
        SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE
    ),
    early AS (
        SELECT content_hash_id,
            SUM(gsc_impressions) AS impr_early,
            SUM(gsc_sum_position) AS sum_pos_early
        FROM daily WHERE report_date <= DATE '2026-03-15'
        GROUP BY content_hash_id
    ),
    late AS (
        SELECT content_hash_id,
            SUM(gsc_impressions) AS impr_late,
            SUM(gsc_sum_position) AS sum_pos_late
        FROM daily WHERE report_date > DATE '2026-03-15'
        GROUP BY content_hash_id
    ),
    labeled AS (
        SELECT
            e.content_hash_id,
            CASE
                WHEN (l.sum_pos_late * 1.0 / NULLIF(l.impr_late,0))
                   > (e.sum_pos_early * 1.0 / NULLIF(e.impr_early,0))
                THEN 1 ELSE 0
            END AS is_declining_label
        FROM early e
        JOIN late l ON e.content_hash_id = l.content_hash_id
        WHERE e.impr_early > 0 AND l.impr_late > 0
    ),
    with_staleness AS (
        SELECT
            lb.content_hash_id,
            lb.is_declining_label,
            DATE '2026-03-15' - dc.content_updated_date AS days_since_update
        FROM labeled lb
        LEFT JOIN read_parquet('{rel}/dim_content.parquet') dc
            ON lb.content_hash_id = dc.content_hash_id
        WHERE dc.content_updated_date IS NOT NULL
    )
    SELECT
        CASE
            WHEN days_since_update < 30 THEN '0-30 days'
            WHEN days_since_update < 90 THEN '30-90 days'
            WHEN days_since_update < 180 THEN '90-180 days'
            ELSE '180+ days'
        END AS staleness_bucket,
        COUNT(*) AS n,
        AVG(is_declining_label) AS decline_rate
    FROM with_staleness
    GROUP BY 1
    ORDER BY MIN(days_since_update)
""")
signal1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────────┬────────┬─────────────────────┐
│ staleness_bucket │   n    │    decline_rate     │
│     varchar      │ int64  │       double        │
├──────────────────┼────────┼─────────────────────┤
│ 0-30 days        │ 140458 │  0.5407096783380085 │
│ 30-90 days       │    169 │  0.4319526627218935 │
│ 90-180 days      │    736 │ 0.44565217391304346 │
│ 180+ days        │    104 │  0.4423076923076923 │
└──────────────────┴────────┴─────────────────────┘

**Signal 2 — CTR vs** **position (bucket table):**


In [19]:
signal2 = con.sql(f"""
    WITH daily AS (
        SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    ),
    page_stats AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impr,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions) AS avg_position
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 50
    )
    SELECT
        CASE
            WHEN avg_position < 3 THEN '1-3'
            WHEN avg_position < 10 THEN '4-10'
            WHEN avg_position < 20 THEN '11-20'
            ELSE '20+'
        END AS position_bucket,
        COUNT(*) AS n,
        AVG(clicks * 1.0 / impr) AS avg_ctr
    FROM page_stats
    GROUP BY 1
    ORDER BY MIN(avg_position)
""")
signal2

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────────────┬───────┬───────────────────────┐
│ position_bucket │   n   │        avg_ctr        │
│     varchar     │ int64 │        double         │
├─────────────────┼───────┼───────────────────────┤
│ 1-3             │ 10893 │  0.003462090137384764 │
│ 4-10            │ 53068 │ 0.0032549734006350876 │
│ 11-20           │ 22182 │  0.002456467517894244 │
│ 20+             │ 29580 │  0.001281075875045669 │
└─────────────────┴───────┴───────────────────────┘

Signal 1 (staleness, days_since_update) — VERDICT: OPPOSITE
Bucket table shows decline_rate is highest in the freshest bucket (0-30 days:
54%) and lower in older buckets (30-90: 43%, 90-180: 45%, 180+: 44%), n=140458/
169/736/104. This contradicts the assumption that older content declines more.
Staleness alone is dropped as a primary rule driver — this negative result
prevents building a rule on a false premise.

Signal 2 (CTR-vs-position) — VERDICT: CONFIRMED
Bucket table shows a clean monotonic drop: position 1-3 → CTR 0.346%, 4-10 →
0.325%, 11-20 → 0.246%, 20+ → 0.128%, n=10893/53068/22182/29580. CTR reliably
degrades as position worsens, confirming this is a real, usable signal.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [20]:
import os
os.makedirs('work/outputs', exist_ok=True)

In [21]:
ranked_queue = con.sql(f"""
    WITH daily AS (
        SELECT * FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-03/data_0.parquet')
        WHERE gsc_data_available IS TRUE AND gsc_impressions > 0
    ),
    page_stats AS (
        SELECT
            content_hash_id,
            SUM(gsc_impressions) AS impr,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) * 1.0 / SUM(gsc_impressions) AS avg_position,
            SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions) AS ctr
        FROM daily
        GROUP BY content_hash_id
        HAVING SUM(gsc_impressions) > 50
    ),
    with_expected_ctr AS (
        SELECT *,
            CASE
                WHEN avg_position < 3 THEN 0.00346
                WHEN avg_position < 10 THEN 0.00325
                WHEN avg_position < 20 THEN 0.00246
                ELSE 0.00128
            END AS expected_ctr
        FROM page_stats
    )
    SELECT
        content_hash_id,
        avg_position,
        ctr,
        expected_ctr,
        impr,
        (expected_ctr - ctr) * impr AS score,
        CASE
            WHEN ctr < expected_ctr * 0.6 THEN 'CTR_UNDERPERFORM'
            ELSE 'HEALTHY'
        END AS reason_code,
        CASE
            WHEN ctr < expected_ctr * 0.6 THEN 'fix-ctr'
            ELSE 'monitor'
        END AS action
    FROM with_expected_ctr
    ORDER BY score DESC
""")

df_queue = ranked_queue.df()

import os
os.makedirs('work/outputs', exist_ok=True)
df_queue.to_csv('work/outputs/baseline_action_score.csv', index=False)
print(f"Wrote {len(df_queue)} rows to CSV")
df_queue.head(20)

Wrote 115723 rows to CSV


,content_hash_id,avg_position,ctr,expected_ctr,impr,score,reason_code,action
0,content_44f34c0a90047651,0.665877,0.000113,0.00346,212404.0,710.91784,CTR_UNDERPERFORM,fix-ctr
1,content_8e1334d6356668e3,2.693038,0.000007,0.00346,134984.0,466.04464,CTR_UNDERPERFORM,fix-ctr
2,content_fec55986a1868d62,0.308426,0.000008,0.00346,124075.0,428.29950,CTR_UNDERPERFORM,fix-ctr
3,content_34a70fea29d15f24,3.166132,0.000301,0.00325,143019.0,421.81175,CTR_UNDERPERFORM,fix-ctr
4,content_8d7d99f109e19aa2,2.468557,0.001420,0.00346,203497.0,415.09962,CTR_UNDERPERFORM,fix-ctr
5,content_7c6373141eae744a,5.948459,0.000626,0.00325,132593.0,347.92725,CTR_UNDERPERFORM,fix-ctr
6,content_f6116743b00afc2d,9.735658,0.000139,0.00325,107584.0,334.64800,CTR_UNDERPERFORM,fix-ctr
7,content_acbcc847f8996314,3.396293,0.001534,0.00325,170808.0,293.12600,CTR_UNDERPERFORM,fix-ctr
8,content_9c057b66c30a3abb,0.116003,0.000012,0.00346,83834.0,289.06564,CTR_UNDERPERFORM,fix-ctr
9,content_cd3d932d4e1c8db0,7.831807,0.000045,0.00325,89332.0,286.32900,CTR_UNDERPERFORM,fix-ctr


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [22]:
top20 = df_queue.head(20).copy()

for i, row in top20.iterrows():
    gap = row['expected_ctr'] - row['ctr']
    confidence = "high" if row['impr'] > 100000 else "medium"
    what_wrong = (
        "position tracking looks off for this page (avg_position < 1, unusual for GSC)"
        if row['avg_position'] < 1
        else "if the low CTR reflects a genuinely irrelevant query mix rather than a fixable on-page issue"
    )
    print(f"{i+1}. {row['content_hash_id']}")
    print(f"   action: {row['action']} | reason_code: {row['reason_code']}")
    print(f"   why: observed CTR ({row['ctr']:.4%}) is far below expected "
          f"({row['expected_ctr']:.4%}) for its position bucket, at {row['impr']:.0f} impressions "
          f"— gap of {gap:.4%}, confidence: {confidence}")
    print(f"   what would make it wrong: {what_wrong}")
    print()

1. content_44f34c0a90047651
   action: fix-ctr | reason_code: CTR_UNDERPERFORM
   why: observed CTR (0.0113%) is far below expected (0.3460%) for its position bucket, at 212404 impressions — gap of 0.3347%, confidence: high
   what would make it wrong: position tracking looks off for this page (avg_position < 1, unusual for GSC)

2. content_8e1334d6356668e3
   action: fix-ctr | reason_code: CTR_UNDERPERFORM
   why: observed CTR (0.0007%) is far below expected (0.3460%) for its position bucket, at 134984 impressions — gap of 0.3453%, confidence: high
   what would make it wrong: if the low CTR reflects a genuinely irrelevant query mix rather than a fixable on-page issue

3. content_fec55986a1868d62
   action: fix-ctr | reason_code: CTR_UNDERPERFORM
   why: observed CTR (0.0008%) is far below expected (0.3460%) for its position bucket, at 124075 impressions — gap of 0.3452%, confidence: high
   what would make it wrong: position tracking looks off for this page (avg_position < 1, unusual

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [23]:
# Weak picks: anomalous position values
weak_picks = df_queue[df_queue['avg_position'] < 1]
print(f"Weak picks with avg_position < 1 (likely data quirk, not a real ranking): {len(weak_picks)} rows")
weak_picks.head(10)

Weak picks with avg_position < 1 (likely data quirk, not a real ranking): 796 rows


,content_hash_id,avg_position,ctr,expected_ctr,impr,score,reason_code,action
0,content_44f34c0a90047651,0.665877,0.000113,0.00346,212404.0,710.91784,CTR_UNDERPERFORM,fix-ctr
2,content_fec55986a1868d62,0.308426,0.000008,0.00346,124075.0,428.29950,CTR_UNDERPERFORM,fix-ctr
8,content_9c057b66c30a3abb,0.116003,0.000012,0.00346,83834.0,289.06564,CTR_UNDERPERFORM,fix-ctr
21,content_c46df0fa61530d86,0.969332,0.000597,0.00346,70398.0,201.57708,CTR_UNDERPERFORM,fix-ctr
34,content_b2b85c287474668d,0.854787,0.000934,0.00346,65304.0,164.95184,CTR_UNDERPERFORM,fix-ctr
96,content_b13e95d379c78818,0.945311,0.001984,0.00346,76121.0,112.37866,CTR_UNDERPERFORM,fix-ctr
234,content_a2ebfe89d44ed4ff,0.560351,0.001298,0.00346,35426.0,76.57396,CTR_UNDERPERFORM,fix-ctr
291,content_a5e1a1f3b30597bf,0.969208,0.000747,0.00346,25429.0,68.98434,CTR_UNDERPERFORM,fix-ctr
433,content_5f37bc769394835c,0.615286,0.000389,0.00346,18003.0,55.29038,CTR_UNDERPERFORM,fix-ctr
512,content_6027078602b6c870,0.998846,0.000787,0.00346,19071.0,50.98566,CTR_UNDERPERFORM,fix-ctr


Weak picks: 796 rows out of 115,723 (0.7%) have avg_position < 1, which is
unusual since GSC positions are 1-indexed. Examples: content_44f34c0a90047651
(0.67), content_fec55986a1868d62 (0.31), content_9c057b66c30a3abb (0.12) —
notably, several of these also rank in the top 20 by score, meaning the
highest-urgency items in the queue include this data quirk. This could be an
aggregation artifact (e.g. averaging gsc_sum_position/gsc_impressions across
days with very low impression counts pulling the mean down) rather than a
genuine ranking signal. Before acting on "fix-ctr" for these specific pages,
I would manually verify the raw daily position values rather than trusting
the aggregate.

For the remaining top-20 picks (not affected by the position quirk), the
main risk is that low CTR could reflect a genuinely irrelevant query mix
(the page ranks but for the wrong intent) rather than a fixable on-page
issue like a weak title/meta description — that distinction isn't
observable from this data alone and would need manual query-level review.

Leakage check: confirmed no future-window data used — all inputs (impr, ctr,
avg_position) are aggregated only from the full month=2026-03 daily table,
independent of any label construction. This rule uses no product flags and
no data from outside the analysis window (the 2026-06 sample file was never
touched here). Score is purely observational/decision-support, not derived
from any outcome label.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.